In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np
import pandas as pd
import os

base_path = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'

for root, dirs, filenames in os.walk(base_path):
    # Calculate depth relative to base_path
    depth = root[len(base_path):].count(os.sep)
    
    # If we are inside a genre folder (e.g., messy_mashup/genres_stems/pop)
    # we empty 'dirs' so it doesn't go into 'pop.00001', 'pop.00002', etc.
    if 'genres_stems' in root and depth >= 1:
        dirs[:] = [] 
        print(f"  📂 {os.path.basename(root)}/ (Contains 100 song subfolders)")
        continue

    # Print files for the root and main directories (like sample_submission.csv)
    rel_path = os.path.relpath(root, base_path)
    if rel_path == ".":
        print(f"📁 Root: {os.path.basename(base_path.strip('/'))}")
    else:
        print(f"  📂 {rel_path}/")
        
    for f in filenames[:2]: # Show only first 2 files in any directory
        print(f"    📄 {f}")
        
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

📁 Root: messy_mashup
    📄 sample_submission.csv
    📄 test.csv
  📂 ESC-50-master/
    📄 LICENSE
    📄 README.md
  📂 ESC-50-master/meta/
    📄 esc50.csv
    📄 esc50-human.xlsx
  📂 ESC-50-master/audio/
    📄 5-257349-A-15.wav
    📄 5-195557-A-19.wav
  📂 genres_stems/ (Contains 100 song subfolders)
  📂 mashups/
    📄 song2501.wav
    📄 song0956.wav


# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


- Complete the function `build_dataset` in question notebook and answer following questions (Q1 to Q3).
Hint: 1kb = 1024 bytes
- What is the value of total number of corrupted sounds ( less than 4kb) + (total number of sounds < 5.0491MB)
- 1256
- What is the absolute difference between  total number of sounds > 5.0493MB and total number of sounds < 5.0491MB ?
- 1072
- What is the absolute difference between the number of training reggae drum samples and the number of validation country vocal samples?
- 66
- Complete the function `find_long_silences` in question notebook and answer following questions (Q4 to Q9).
- Total number of sound files having silence greater than equal to 5 secs.
- 676
- Total number of sound tracks in Vocals where silence >= 5 secs
- 304
- What's the average Silence Length in Vocals (in secs).
- 12.79
- Total number of drums sound tracks in jazz where silence >= 5 secs
- 21
- Total number of drums sound tracks in jazz where silence >= 5 secs and Silence_Location is only middle.
- 15
- Total number of drums sound tracks in jazz where silence >= 5 secs and Max_Silence_Sec >= 10.
- 7
- Select the first song from the ‘rock’ genre, combine all stems to prepare a sample, perform the tasks outlined in the notebook, and answer questions 10–12 based on the results.
- What is the length of the mix sample?
- 110250
- What is the value of RMS Amplitude of mix sample?
- 0.11
- What is the value of max value of peak  normalized sample ?
- 0.89

In [3]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [4]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [5]:
# CONFIGURATION
DATA_ROOT = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
GENRES = ["blues", "classical", "country", "disco", "hiphop",
"jazz", "metal", "pop", "reggae", "rock"]
STEMS = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0

In [6]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)
    
    # Counters for Q1 & Q2
    corrupted_sounds = 0
    sounds_lt_5_0491_mb = 0
    sounds_gt_5_0493_mb = 0
    
    # Size references (1kb = 1024 bytes)
    kb_4 = 4 * 1024
    mb_5_0491 = 5.0491 * 1024 * 1024
    mb_5_0493 = 5.0493 * 1024 * 1024

    for genre in tqdm(GENRES, desc="Processing Genres"):
        genre_path = os.path.join(root_dir, genre)
        if not os.path.exists(genre_path):
            continue
        
        song_folders = sorted([f for f in os.listdir(genre_path) if os.path.isdir(os.path.join(genre_path, f))])
        valid_songs = []
        
        for song_folder in song_folders:
            song_path = os.path.join(genre_path, song_folder)
            stems_present = [os.path.join(song_path, stem) for stem in STEMS]
            
            if not all(os.path.exists(stem_path) for stem_path in stems_present):
                continue
            
            corrupted = False
            for stem_path in stems_present:
                file_size = os.path.getsize(stem_path)
                
                # Update Counters for Q1 & Q2
                if file_size < kb_4:
                    corrupted_sounds += 1
                    corrupted = True
                if file_size < mb_5_0491:
                    sounds_lt_5_0491_mb += 1
                if file_size > mb_5_0493:
                    sounds_gt_5_0493_mb += 1
            
            if not corrupted:
                valid_songs.append(song_folder)
        
        rng.shuffle(valid_songs)
        split_idx = int(len(valid_songs) * (1 - val_split))
        train_songs = valid_songs[:split_idx]
        val_songs = valid_songs[split_idx:]
        
        def add_to_dict(target_dict, song_list):
            for song_folder in song_list:
                song_path = os.path.join(genre_path, song_folder)
                for stem in STEMS:
                    stem_name = stem.replace('.wav', '')
                    stem_path = os.path.join(song_path, stem)
                    target_dict[genre][stem_name].append(stem_path)
        
        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)

    print("\n" + "="*60)
    print("ANSWERS FOR Q1-Q3")
    print("="*60)
    
    q1_ans = corrupted_sounds + sounds_lt_5_0491_mb
    q2_ans = abs(sounds_gt_5_0493_mb - sounds_lt_5_0491_mb)
    print(f"[Q1] Total corrupted (<4kb) + total sounds (<5.0491MB): {q1_ans}")
    print(f"[Q2] Absolute difference (>5.0493MB and <5.0491MB): {q2_ans}")
    
    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)

q3_ans = abs(len(tr['reggae']['drums']) - len(val['country']['vocals']))
print(f"[Q3] Absolute diff (train reggae drums vs val country vocals): {q3_ans}")
print("="*60)

Processing Genres: 100%|██████████| 10/10 [00:05<00:00,  1.72it/s]


ANSWERS FOR Q1-Q3
[Q1] Total corrupted (<4kb) + total sounds (<5.0491MB): 1256
[Q2] Absolute difference (>5.0493MB and <5.0491MB): 1072
[Q3] Absolute diff (train reggae drums vs val country vocals): 66


In [7]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    records = []
    total_files = sum(len(paths) for genre_data in dataset_dict.values() for paths in genre_data.values())
    pbar = tqdm(total=total_files, desc="Processing files")
    
    for genre, stems_dict in dataset_dict.items():
        for stem_name, file_paths in stems_dict.items():
            for file_path in file_paths:
                try:
                    y, _ = librosa.load(file_path, sr=sr, duration=None)
                    total_duration = len(y) / sr
                    non_silent_intervals = librosa.effects.split(y, top_db=top_db)
                    
                    max_silence = 0.0
                    silence_type = []
                    
                    if len(non_silent_intervals) == 0:
                        max_silence = total_duration
                        silence_type = ["start", "middle", "end"]
                    else:
                        start_silence = non_silent_intervals[0][0] / sr
                        if start_silence > max_silence:
                            max_silence = start_silence
                            silence_type = ["start"]
                        
                        end_silence = (len(y) - non_silent_intervals[-1][1]) / sr
                        if end_silence > max_silence:
                            max_silence = end_silence
                            silence_type = ["end"]
                        elif abs(end_silence - max_silence) < 0.01:
                            if "end" not in silence_type: silence_type.append("end")
                        
                        for i in range(len(non_silent_intervals) - 1):
                            gap_duration = (non_silent_intervals[i + 1][0] - non_silent_intervals[i][1]) / sr
                            if gap_duration > max_silence:
                                max_silence = gap_duration
                                silence_type = ["middle"]
                            elif abs(gap_duration - max_silence) < 0.01:
                                if "middle" not in silence_type: silence_type.append("middle")
                    
                    if max_silence >= threshold_sec:
                        records.append({
                            "Genre": genre,
                            "Stem": stem_name,
                            "Duration": round(total_duration, 2),
                            "Max_Silence_Sec": round(max_silence, 2),
                            "Silence_Location": ", ".join(silence_type),
                            "File_Path": file_path
                        })
                except Exception as e:
                    pass
                pbar.update(1)
    
    pbar.close()
    return pd.DataFrame(records)

df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

print("\n" + "="*60)
print("ANSWERS FOR Q4-Q9")
print("="*60)
print(f"[Q4] Total files having silence >= 5 secs: {len(df_silence)}")

df_vocals = df_silence[df_silence['Stem'] == 'vocals']
print(f"[Q5] Total tracks in Vocals where silence >= 5 secs: {len(df_vocals)}")
print(f"[Q6] Average Silence Length in Vocals (secs): {round(df_vocals['Max_Silence_Sec'].mean(), 2)}")

df_jazz_drums = df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums')]
print(f"[Q7] Total drums tracks in jazz where silence >= 5 secs: {len(df_jazz_drums)}")

q8_ans = len(df_jazz_drums[df_jazz_drums['Silence_Location'] == 'middle'])
print(f"[Q8] ...and Silence_Location is only middle: {q8_ans}")

q9_ans = len(df_jazz_drums[df_jazz_drums['Max_Silence_Sec'] >= 10])
print(f"[Q9] ...and Max_Silence_Sec >= 10: {q9_ans}")
print("="*60)

Processing files: 100%|██████████| 3320/3320 [07:28<00:00,  7.40it/s]


ANSWERS FOR Q4-Q9
[Q4] Total files having silence >= 5 secs: 680
[Q5] Total tracks in Vocals where silence >= 5 secs: 304
[Q6] Average Silence Length in Vocals (secs): 12.59
[Q7] Total drums tracks in jazz where silence >= 5 secs: 24
[Q8] ...and Silence_Location is only middle: 16
[Q9] ...and Max_Silence_Sec >= 10: 7


In [8]:
# Select EXACTLY the first song from the 'rock' genre folder
rock_path = os.path.join(DATA_ROOT, 'rock')
first_rock_song = sorted([f for f in os.listdir(rock_path) if os.path.isdir(os.path.join(rock_path, f))])[0]
first_rock_song_path = os.path.join(rock_path, first_rock_song)

stems_audio = []
try:
    for key in STEM_KEYS:
        stem_path = os.path.join(first_rock_song_path, f"{key}.wav")
        # Load exactly 5.0s length for consistency based on standard configuration
        y, sr_loaded = librosa.load(stem_path, sr=SR, duration=DURATION)
        stems_audio.append(y)
        print(f"Loaded {key}: {stem_path}")
    print("\nAudio loaded successfully.")
except Exception as e:
    print(f"ERROR: {e}")

Loaded drums: /kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00000/drums.wav
Loaded vocals: /kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00000/vocals.wav
Loaded bass: /kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00000/bass.wav
Loaded other: /kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00000/other.wav

Audio loaded successfully.


In [9]:
print("\n" + "="*60)
print("ANSWERS FOR Q10-Q12")
print("="*60)

# Stack them into a numpy array (Shape: 4 x Samples)
stems_stack = np.array(stems_audio)

# Mix the stems by summing them element-wise
mix_raw = np.sum(stems_stack, axis=0)

# Q10: Length of the mix sample
q10_len = len(mix_raw)
print(f"[Q10] Length of the mix sample: {q10_len}")

# Q11: Calculate RMS Amplitude MANUALLY
rms_val = np.sqrt(np.mean(mix_raw**2))
print(f"[Q11] RMS Amplitude of mix sample: {round(rms_val, 2)}")

# Q12: Peak Normalization
# Note: Peak normalization scales by dividing by the absolute max, meaning np.max(np.abs(mix_norm)) == 1.0. 
# However, to get the specific MAXIMUM raw numerical positive peak inside that scaled array, we use np.max() instead of absolute.
max_abs_val = np.max(np.abs(mix_raw))

if max_abs_val > 0:
    mix_norm = mix_raw / max_abs_val
else:
    mix_norm = mix_raw

q12_max_peak = np.max(mix_norm)
print(f"[Q12] Max positive value of peak normalized sample: {round(q12_max_peak, 2)}")
print("="*60)

# VALIDATION
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."


ANSWERS FOR Q10-Q12
[Q10] Length of the mix sample: 110250
[Q11] RMS Amplitude of mix sample: 0.20000000298023224
[Q12] Max positive value of peak normalized sample: 0.9900000095367432
